In [1]:
import numpy as np
import sys
import h5py
from scipy.stats import rankdata
import scipy.stats as stats

**Preprocessing**

Constants

In [2]:
sfc_filepath = '/home/ctrader/hilbert/results/DHS/SFC/20230622/order_6_windows_annotation_sfc_scores.txt.gz'

num_factors = 16
convert2paper_order = np.array([8, 3, 5, 1, 4, 15, 9, 6, 2, 12, 13, 11, 0, 10, 7, 14])

# for search space
multi_factor_thresh = 2
top_region_thresh = 21

Load data

In [3]:
# load factor names
factors = np.loadtxt(
    sfc_filepath, 
    max_rows=1, 
    usecols=np.arange(3, 3 + num_factors), 
    dtype=str, 
    delimiter='\t'
)[convert2paper_order]

# load SFC scores from all factors and segments
sfc_scores = np.loadtxt(
    sfc_filepath, 
    skiprows=1, 
    usecols=np.arange(3, 3 + num_factors), 
    dtype=float, 
    delimiter='\t'
)[:, convert2paper_order]

# set -1 values to 0
sfc_scores = np.where(sfc_scores < 0, 0, sfc_scores)

# load segment coordinates
coordinates = np.loadtxt(
    sfc_filepath, 
    skiprows=1, 
    usecols=np.arange(0, 3), 
    dtype=str, 
    delimiter='\t'
)

Rank data

In [4]:
# Function to take our sfc scores and rank them from 0 to 1
def rank_regions(arr):
    # rank each region
    ranked_regions = rankdata(arr, axis=0, method='min') - 1

    # normalize by number of segments
    ranks = ranked_regions / (ranked_regions.shape[0] - 1)

    return ranks

sfc_ranks = rank_regions(sfc_scores)

Define Search Space

In [5]:
# calculate the number of factors significant in each segment
segment_counts = np.count_nonzero(sfc_ranks, axis=1)

# include any segment with at least <multi_factor_thresh> counts
segments_to_include = np.where(segment_counts >= multi_factor_thresh)[0]

# segments we may include
potential_segments_to_include = np.where(
    (segment_counts < multi_factor_thresh) &
    (segment_counts > 0)
)[0]

# sfc cutoff values for each factor
sfc_thresh = np.sort(sfc_ranks, axis=0)[-top_region_thresh]

# potential segments greater than sfc threshold
mask = (
    (sfc_ranks[potential_segments_to_include] >= sfc_thresh) &
    (sfc_ranks[potential_segments_to_include] > 0)
)

# limit potential segments
potential_segments_to_include = potential_segments_to_include[
    np.where(
        np.count_nonzero(mask, axis=1) > 0
    )[0]
]

# full list of segments to include
top_inds = np.unique(
    np.append(segments_to_include, potential_segments_to_include)
)

# find corresponding ranks/coordinates
search_space_ranks = sfc_ranks[top_inds]
search_space_coordinates = coordinates[top_inds]

**Narration & Simsearch**

Functions

In [6]:
# determine roi from query coordinates
def load_roi(genomic_location, coordinates, sfc_ranks, tabix_path=''):
    # determine chromosome, start, and end from query
    query_start_end = genomic_location.split(':')[1].split('-')
    query_chrm = genomic_location.split(':')[0]
    query_start = int(query_start_end[0])
    query_end = int(query_start_end[1])

    # retrieve possible rois
    if tabix_path:  # for sandbox, if we have the tabix path
        possible_coords = tabix_query(tabix_path, genomic_location)
    else:  # for sandbox, otherwise just search over coordinates
        query_inds = np.where(
            (coordinates[:, 0] == query_chrm) & 
            (coordinates[:, 1].astype(int) < query_end) & 
            (coordinates[:, 2].astype(int) > query_start)
        )[0]

        possible_coords = np.hstack((
            np.hstack((
                coordinates[query_inds], query_inds.reshape(-1, 1).astype(str)
            )), 
            sfc_ranks[query_inds].astype(str)
        ))

    # find closest roi
    start_dist = np.abs(possible_coords[:, 1].astype(int) - query_start)
    end_dist = np.abs(possible_coords[:, 2].astype(int) - query_end)
    total_dist = start_dist + end_dist
    min_dist = np.min(total_dist)
    min_dist_region_inds = np.where(total_dist == min_dist)[0]
    middle_ind = min_dist_region_inds[np.floor(len(min_dist_region_inds) / 2).astype(int)]
    query_region = possible_coords[middle_ind]

    roi = int(query_region[3])
    roi_ranks = query_region[4:].astype('float32')
    roi_coords = query_region[:3]

    return roi, roi_ranks, roi_coords

# find the factors we want to search over
def setup_selected_factors(selected_factors_str):
    selected_factors = np.int_(
        selected_factors_str.strip('[').strip(']').split(',')
    )
    return selected_factors

# function to order our factors. We order selected factors based on the sum of ranks
# in our top regions. We order unselected factors based on their sums in all
# regions that also contain our selected factors
def determine_factor_order(top_regions, selected_factors, search_space_ranks):
    top_region_ranks = search_space_ranks[top_regions]

    # order selected factors first
    selected_factors_best_order = selected_factors[
        np.argsort(
            -np.sum(
                top_region_ranks[:, selected_factors],
                axis=0
            )
        )
    ]

    # order remaining factors
    # find regions that are significant for our selected factors
    regions_to_consider = np.where(
        np.count_nonzero(
            search_space_ranks[:, selected_factors_best_order],
            axis=1
        ) > 0
    )[0]

    total_factors = np.arange(search_space_ranks.shape[1])
    # unselected factors
    leftover_factors = total_factors[~np.isin(total_factors, selected_factors_best_order)]
    # order leftover factors based on sum of ranks in our considered regions
    leftover_factors_best_order = leftover_factors[
        np.argsort(
            -np.sum(
                search_space_ranks[regions_to_consider][:, leftover_factors],
                axis=0
            ),
            kind='stable'
        )
    ]

    factor_order = np.append(selected_factors_best_order, leftover_factors_best_order)
    return factor_order

# search for the best regions for selected factors
def search_over_selected_factors(num_regions, selected_factors, search_space_ranks, search_space_coordinates):
    # focus on selected factors
    selected_factor_search_space_ranks = search_space_ranks[:, selected_factors]

    # limit regions to just those containing all factors
    selected_factor_region_inds = np.where(
        np.count_nonzero(selected_factor_search_space_ranks, axis=1) == len(selected_factors)
    )[0]
    selected_factor_search_space_filtered_ranks = selected_factor_search_space_ranks[selected_factor_region_inds]

    # fusion
    selected_factor_search_space_score = 1 - np.prod(1 - selected_factor_search_space_filtered_ranks, axis=1)

    # get the best regions
    top_filtered_regions = np.argsort(-selected_factor_search_space_score)[:num_regions]
    top_regions = selected_factor_region_inds[top_filtered_regions]

    # factor order
    factor_order = determine_factor_order(top_regions, selected_factors, search_space_ranks)

    # find info for each top region
    top_regions_ranks = search_space_ranks[top_regions][:, factor_order]
    top_regions_coords = search_space_coordinates[top_regions]
    top_regions_score = selected_factor_search_space_score[top_filtered_regions]
    
    return top_regions, top_regions_ranks, top_regions_coords, top_regions_score, factor_order

# find the factors we want to search over. This is based on the ranks of the 
# query region. We only search over the factors the query region ranks > 
# <query_factor_thresh> in
def refine_factors_to_search_over(roi_ranks, query_factor_thresh):
    query_factors_above_thresh = np.where(roi_ranks > query_factor_thresh)[0]
    # if no factors above threshold, search best one
    if len(query_factors_above_thresh) == 0:
        sort_order = np.argsort(-roi_ranks)
        included_factors = sort_order[:1]
    else:
        included_factors = query_factors_above_thresh
    return included_factors

# regional simsearch. Finds the best regions for the factors that the query region ranks well in.
def perform_simsearch(roi_ranks, search_space_ranks, roi, roi_coords, coords, target_factors, num_similar_regions):

    # factors the query region ranks highest in
    query_factor_order = np.argsort(-roi_ranks)

    # convolution
    roi_ranks_targeted = roi_ranks[target_factors]
    # normal convolution
    # roi_conv_all = search_space_ranks[:, target_factors] * roi_ranks_targeted
    # penalized convolution
    roi_conv_all = ((search_space_ranks[:, target_factors] * 2) - 1) * ((roi_ranks_targeted * 2) - 1)

    # fusion
    roi_sim_all = []
    for dl in range(1, len(target_factors) + 1):
        roi_sim = 1 - np.prod(1 - roi_conv_all[:, :dl], axis=1)
        roi_sim_all.append(roi_sim)
    roi_sim_all = np.array(roi_sim_all)

    # most similar regions
    top_regions = np.zeros(
        (len(included_factors), num_similar_regions + 1), 
        dtype=int
    )

    # top region genomic coordinates
    top_region_coords = np.ones(
        (len(included_factors), num_similar_regions + 1, 3),
        dtype=object
    ) * -1

    # add roi to top region lists
    top_regions[:, 0] = roi
    top_region_coords[:, 0] = roi_coords

    # adding similar regions
    sorted_sim_inds_all = np.argsort(-roi_sim_all, axis=1)

    # only add regions if no overlapping regions are already added
    for i in range(len(target_factors)):
        regions_added = 0
        for ind in sorted_sim_inds_all[i]:
            ind_coords = search_space_coordinates[ind]
            num_overlaps = np.count_nonzero(
                (top_region_coords[i][:, 0].astype(str) == ind_coords[0]) &
                (
                    (
                        (top_region_coords[i][:, 1].astype(int) > int(ind_coords[1])) &
                        (top_region_coords[i][:, 1].astype(int) < int(ind_coords[2]))
                    ) |
                    (
                        (top_region_coords[i][:, 2].astype(int) > int(ind_coords[1])) &
                        (top_region_coords[i][:, 2].astype(int) < int(ind_coords[2]))
                    ) |
                    (
                        (top_region_coords[i][:, 1].astype(int) == int(ind_coords[1])) &
                        (top_region_coords[i][:, 2].astype(int) == int(ind_coords[2]))
                    )
                )
            )
            if num_overlaps == 0:
                top_regions[i, regions_added + 1] = ind
                top_region_coords[i, regions_added + 1] = ind_coords
                regions_added += 1
            if regions_added == num_similar_regions:
                break

    rows = (
        np.ones((len(target_factors), num_similar_regions), dtype=int) *
        np.arange(len(target_factors)).reshape(-1, 1)
    )
    return_sim = np.hstack((
        np.ones(len(target_factors)).reshape(-1, 1),
        roi_sim_all[rows, top_regions[:, 1:]]
    ))
    return_ranks = np.hstack((
        np.array([roi_ranks.reshape(1, -1)] * len(target_factors)),
        search_space_ranks[top_regions[:, 1:]]
    ))

    return (top_regions, top_region_coords, return_sim, return_ranks,
            query_factor_order, target_factors)

Narration

In [7]:
# query information 
genomic_location = 'chr1:1000000-2000000'

# narration
roi, roi_ranks, roi_coords = load_roi(genomic_location, coordinates, sfc_ranks)

Multi-Factor Search

In [8]:
# query information
selected_factors_str = '[1,5,10,12]'
num_regions = 20

# multi-factor search
selected_factors = setup_selected_factors(selected_factors_str)
top_regions, top_regions_ranks, top_regions_coords, top_regions_score, factor_order = search_over_selected_factors(num_regions, selected_factors, search_space_ranks, search_space_coordinates)

Regional Simsearch

In [9]:
# query information 
genomic_location = 'chr1:1000000-2000000'
query_factor_thresh = 0.5
num_similar_regions = 20

# narration
roi, roi_ranks, roi_coords = load_roi(genomic_location, coordinates, sfc_ranks)
included_factors = refine_factors_to_search_over(roi_ranks, query_factor_thresh)

(top_regions, top_region_coords, top_regions_sim,
 top_regions_ranks, query_factor_order,
 target_factors) = perform_simsearch(roi_ranks, search_space_ranks, roi, roi_coords, search_space_coordinates, included_factors, num_similar_regions)